# Laboratorio Cuantitativo Aplicado · Sesión 9 · Solución (Python)

Versión resuelta, para calificar. Integra la Sesión 7 (diseño, inferencia, agregación) y la 8 (reproducibilidad) sobre un microdato nuevo. Mismos números que la versión en R.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

np.random.seed(909)   # Parte 5: el azar, fijo desde el inicio

d = pd.read_csv("datos_encuesta_s9.csv")
d["estrato"] = pd.Categorical(d["estrato"], categories=["Alta", "Media", "Baja"], ordered=True)
print(len(d), "hogares,", d["upm"].nunique(), "UPM")

900 hogares, 90 UPM


## Parte 1 · Explorar y ponderar

In [2]:
crudo = d["ingreso"].mean()
pond  = np.average(d["ingreso"], weights=d["factor"])
print(f"Ingreso crudo: {crudo:,.0f} | ponderado: {pond:,.0f} | subestima {100*(pond-crudo)/pond:.1f}%")

Ingreso crudo: 7,684 | ponderado: 8,759 | subestima 12.3%


**Justificación.** El crudo (7,684) subestima el poblacional (8,759) en 12.3% porque la muestra sobre-representa Alta marginación (44% de la muestra), el estrato de menor ingreso. El factor de expansión devuelve a cada estrato su peso real. (El error estándar de diseño de esta media se obtiene con una librería de encuestas; aquí la lección de inferencia va en la Parte 2, sobre el efecto.)

## Parte 2 · El efecto con el error estándar correcto

In [3]:
m_iid = smf.ols("score ~ programa", data=d).fit()
m_clu = smf.ols("score ~ programa", data=d).fit(cov_type="cluster", cov_kwds={"groups": d["upm"]})
print(f"Efecto: {m_iid.params['programa']:.1f}")
print(f"EE clasico (IID): {m_iid.bse['programa']:.2f} (p = {m_iid.pvalues['programa']:.3f})")
print(f"EE por conglomerado: {m_clu.bse['programa']:.2f} (p = {m_clu.pvalues['programa']:.3f})")

Efecto: -3.7
EE clasico (IID): 1.02 (p = 0.000)
EE por conglomerado: 1.50 (p = 0.013)


**Justificación.** El punto no cambia (−3.7); el error clásico (1.02) ignora que los hogares se agrupan en UPM y subestima la incertidumbre casi a la mitad. El error por conglomerado (1.50) es el honesto, y deja el efecto apenas significativo. (El signo negativo, además, es engañoso: se corrige en la Parte 3.)

## Parte 3 · Condicionar por estrato (Simpson)

In [4]:
print(d.groupby("programa", observed=True)["score"].agg(["mean", "size"]).round(1))
print()
print(d.groupby(["estrato", "programa"], observed=True)["score"].agg(["mean", "size"]).round(1))

          mean  size
programa            
0         64.8   462
1         61.0   438

                  mean  size
estrato programa            
Alta    0         42.7   101
        1         53.9   299
Media   0         60.4   156
        1         70.7    94
Baja    0         78.9   205
        1         88.2    45


**Justificación.** En agregado, los tratados puntúan por debajo (61.0 vs 64.8): el programa parece dañar. Al condicionar por estrato, sube alrededor de 10 puntos en los tres niveles (Alta +11, Media +10, Baja +9). Se invierte porque el programa se concentró en Alta marginación (299 de 438 tratados), el estrato de menor base. Condicionando, el programa ayuda.

## Parte 4 · Auditar a la IA

Este es el análisis que entregó un asistente. Corre y se ve profesional, pero comete los tres errores de la Sesión 7.

In [5]:
# --- Analisis que entrego la IA (con sus tres errores) ---
ia_ingreso = d["ingreso"].mean()                          # (1) promedio crudo
ia = smf.ols("score ~ programa", data=d).fit()            # (2) EE clasico, (3) agregado
print(f"IA reporta: ingreso promedio {ia_ingreso:,.0f}; el programa cambia el score en "
      f"{ia.params['programa']:.1f} (EE {ia.bse['programa']:.2f})")
# Conclusion de la IA: "ingreso 7,684; el programa REDUCE el score 3.7 puntos, significativo." 

IA reporta: ingreso promedio 7,684; el programa cambia el score en -3.7 (EE 1.02)


In [6]:
# --- La correccion: los tres arreglos ---
print(f"1. Ingreso ponderado: {np.average(d['ingreso'], weights=d['factor']):,.0f}  (no {ia_ingreso:,.0f})")
print(f"2. EE por conglomerado: {m_clu.bse['programa']:.2f}  (no {ia.bse['programa']:.2f})")
print("3. Efecto por estrato (no el agregado):")
for e in ["Alta", "Media", "Baja"]:
    sub = d[d["estrato"] == e]
    dif = sub.loc[sub.programa == 1, "score"].mean() - sub.loc[sub.programa == 0, "score"].mean()
    print(f"   {e}: {dif:+.1f}")

1. Ingreso ponderado: 8,759  (no 7,684)
2. EE por conglomerado: 1.50  (no 1.02)
3. Efecto por estrato (no el agregado):
   Alta: +11.2
   Media: +10.3
   Baja: +9.3


**Auditoría.** La IA cometió los tres errores a la vez. **(1)** Promedió el ingreso sin ponderar: 7,684 cuando el poblacional es 8,759. **(2)** Usó el error estándar clásico (1.02); el correcto por conglomerado es 1.50. **(3)** Leyó el efecto agregado (−3.7) y concluyó que el programa daña; por estrato el efecto es positivo en los tres niveles: el programa ayuda. Nada de esto lo señaló una advertencia: el código corrió sin error. Lo que le faltó no era estadística, sino saber que es una encuesta con diseño y que el programa se focalizó en Alta marginación.

## Parte 5 · Reproducible

In [7]:
import sys, statsmodels, platform
print("Insumo: datos_encuesta_s9.csv,", d.shape[0], "filas x", d.shape[1], "columnas")
print("Semilla fija: np.random.seed(909) al inicio")
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__,
      "| statsmodels", statsmodels.__version__)

Insumo: datos_encuesta_s9.csv, 900 filas x 6 columnas
Semilla fija: np.random.seed(909) al inicio
Python 3.12.3 | pandas 3.0.2 | numpy 2.4.4 | statsmodels 0.15.0


**Justificación.** La semilla (`np.random.seed(909)`, arriba) hace reproducible cualquier remuestreo; el insumo está fijo y descrito; la última celda registra las versiones. La prueba: reiniciar el kernel y correr todo de cero da estos mismos números. El entregable es este notebook, no el prompt que se le dio a la IA.

## Cierre

Un asistente escribe el análisis en segundos, pero aquí cometió los tres errores de diseño y ninguno se notó al correr. Ponderar, conglomerar, condicionar y fijar el azar y el entorno: eso lo decide y lo firma el analista. La IA es el copiloto; el criterio es humano.